In [ ]:
import os
from cv2 import phase
import numpy as np
import plotly
import imageio.v3 as imageio
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

from gvs_propagator.constants import nm, um, mm, cm
from gvs_propagator.partially_coherent import (
    compute_complex_coherence_factor,
    compute_angular_spectrum_gvs,
)
from gvs_propagator.utils import crop_center
from gvs_propagator.masks import create_circular_mask

plotly.offline.init_notebook_mode()


# PARAMETERS

num_pixels_x = ## Specify number of pixels in x-direction ##
num_pixels_y = ## Specify number of pixels in y-direction ##

wavelength      = ## Specify wavelength in meters ##
z_source_object = ## Specify source-to-object distance in meters ##

source_radii = ## Specify source radii in meters ##

coh_names = ## Specify coherence names corresponding to source_radii ##

dx_camera = dy_camera = ## Specify camera pixel size in meters ##
pad_factor = ## Specify padding factor ##


propagation_distances = ## Specify propagation distances in meters ##

objects_root = ## Specify path to the object dataset root directory ##

object_groups = ## [Specify object classes] ##

object_ids = ## [Specify object IDs] ##


# HELPERS

def resize_image_to_grid(image):

    return zoom(
        image,
        (
            num_pixels_y / image.shape[0],
            num_pixels_x / image.shape[1]
        ),
        order=1,
        mode="nearest",
        prefilter=False
    )


def load_object_image(obj_path):


    raw = imageio.imread(obj_path)

    # Convert RGB/RGBA to grayscale if needed
    if raw.ndim == 3:
        raw = raw[..., :3]
        raw = (
            0.2989 * raw[..., 0] +
            0.5870 * raw[..., 1] +
            0.1140 * raw[..., 2]
        )

    # Datatype-based scaling
    if np.issubdtype(raw.dtype, np.integer):
        raw01 = raw.astype(np.float32) / np.iinfo(raw.dtype).max
    else:
        raw01 = raw.astype(np.float32)
        if raw01.max() > 1.0:
            raw01 = raw01 / 255.0

    raw01 = np.clip(raw01, 0.0, 1.0)

    amp = resize_image_to_grid(raw01)
    amp = np.clip(amp, 0.0, 1.0).astype(np.float32)

    return raw01, amp


def normalize_for_saving(image, eps=1e-12):

    image = np.asarray(image)
    image = np.real(image).astype(np.float32)

    mn = np.nanmin(image)
    mx = np.nanmax(image)

    if not np.isfinite(mn) or not np.isfinite(mx):
        raise ValueError("Image contains NaN or Inf values.")

    if (mx - mn) < eps:
        return np.zeros_like(image, dtype=np.float32)

    return (image - mn) / (mx - mn)


def save_true_size_png(path, img01):

    img01 = np.clip(img01, 0.0, 1.0)
    img8 = (img01 * 255.0 + 0.5).astype(np.uint8)
    imageio.imwrite(path, img8)


def show_loaded_object(raw01, amp, obj_group, obj_id):

    plt.figure(figsize=(8, 4), dpi=120)

    plt.subplot(1, 2, 1)
    plt.imshow(raw01, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Raw image: {obj_group}/{obj_id}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(amp, cmap="gray", vmin=0, vmax=1)
    plt.title("Resized object used")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


# MAIN FUNCTION

def show_diffraction():

    out_dir = ## Specify output directory for simulated measurements ## 
    os.makedirs(out_dir, exist_ok=True)

    last_simulated_set = None
    last_display_title = None

    for obj_group in object_groups:
        for obj_id in object_ids:

            obj_path = os.path.join(objects_root, obj_group, f"{obj_id}.png")

            if not os.path.exists(obj_path):
                print(f"Missing file: {obj_path}")
                continue

            
            raw01, amp = load_object_image(obj_path)

            print(
                f"{obj_group}/{obj_id} | "
                f"raw min={raw01.min():.4f}, max={raw01.max():.4f}, "
                f"mean={raw01.mean():.4f}, std={raw01.std():.4f} | "
                f"amp min={amp.min():.4f}, max={amp.max():.4f}, "
                f"mean={amp.mean():.4f}, std={amp.std():.4f}"
            )

            if amp.std() < 1e-4:
                print(f"WARNING: {obj_group}/{obj_id} is almost flat after loading/resizing.")


            
            phase = amp * np.pi
            obj_complex = np.exp(1j * phase).astype(np.complex64)

            # Padding
            pad_y = pad_factor * num_pixels_y
            pad_x = pad_factor * num_pixels_x

            obj_padded = np.pad(
                obj_complex,
                ((pad_y, pad_y), (pad_x, pad_x)),
                mode="constant",
                constant_values=1
            )

            for z in propagation_distances:

                ny, nx = obj_padded.shape

                dx_obj = (wavelength * z) / (2 * dx_camera * nx)
                dy_obj = (wavelength * z) / (2 * dy_camera * ny)

                dx_src = (wavelength * z_source_object) / (2 * dx_obj * nx)
                dy_src = (wavelength * z_source_object) / (2 * dy_obj * ny)

                sim_set = []

                for k, r in enumerate(source_radii):

                    src_I = create_circular_mask(nx, dx_src, r, ny, dy_src)
                    gamma = compute_complex_coherence_factor(src_I)

                    raw_Igvs = crop_center(
                        compute_angular_spectrum_gvs(
                            obj_padded,
                            gamma,
                            z_source_object,
                            z,
                            dx_camera,
                            wavelength
                        ),
                        num_pixels_y,
                        num_pixels_x
                    )

                    print(
                        f"    z={z/mm:.2f} mm, source={coh_names[k]} um | "
                        f"raw Igvs min={np.real(raw_Igvs).min():.6e}, "
                        f"max={np.real(raw_Igvs).max():.6e}, "
                        f"std={np.real(raw_Igvs).std():.6e}"
                    )

                    Igvs = normalize_for_saving(raw_Igvs)
                    sim_set.append(Igvs)

                    subdir = os.path.join(out_dir, obj_group, obj_id, coh_names[k])
                    os.makedirs(subdir, exist_ok=True)


                    save_true_size_png(
                    os.path.join(subdir, f"Simulated_{z/mm:.0f}mm.png"),
                    Igvs
                    )

                last_simulated_set = sim_set
                last_display_title = f"{obj_group}/{obj_id}, z = {z/mm:.2f} mm"

    print(f"Saved simulated measurements to: {os.path.abspath(out_dir)}")

    # Display the final simulated images
    if last_simulated_set is not None:

        fig, axes = plt.subplots(2, 5, figsize=(16, 6), dpi=150)
        axes = axes.ravel()

        for ax, im, title in zip(axes, last_simulated_set, coh_names):
            ax.imshow(im, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"{title} um", fontsize=10)
            ax.axis("off")

        plt.suptitle(last_display_title, fontsize=12)
        plt.tight_layout()
        plt.show()


# Run
show_diffraction()